# 08 — Z(t) aux points de Gram comme features ML

Dans le notebook 07, on utilisait les **positions** des zéros précédents comme features.  
Ici on ajoute les **valeurs de Z(t) aux points de Gram** — la structure locale de ζ autour de chaque intervalle.  
C'est l'approche utilisée dans la littérature récente sur le sujet.

In [1]:
import numpy as np
import struct
import mpmath
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import Ridge

mpmath.mp.dps = 15

TEMPLATE     = "plotly_dark"
COLOR_MAIN   = "#7C6FCD"
COLOR_ACCENT = "#F2A623"
COLOR_MUTED  = "#888780"
COLOR_DANGER = "#E24B4A"
COLOR_TEAL   = "#1D9E75"

def parse_zeros_dat(filepath):
    with open(filepath, "rb") as f:
        raw = f.read()
    pos = 0
    B = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
    all_zeros = []
    for block in range(B):
        t0  = struct.unpack_from('<d', raw, pos)[0]; pos += 8
        t1  = struct.unpack_from('<d', raw, pos)[0]; pos += 8
        Nt0 = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
        Nt1 = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
        n_zeros = Nt1 - Nt0
        cumsum = 0
        for _ in range(n_zeros):
            lo  = struct.unpack_from('<Q', raw, pos)[0]; pos += 8
            mid = struct.unpack_from('<I', raw, pos)[0]; pos += 4
            hi  = struct.unpack_from('<B', raw, pos)[0]; pos += 1
            z = (hi << 96) + (mid << 64) + lo
            cumsum += z
            all_zeros.append(t0 + cumsum * 2**-101)
    return np.array(all_zeros)

def compute_gram_and_Z(t_min, t_max):
    n_start = int(np.ceil(float(mpmath.siegeltheta(t_min))  / np.pi))
    n_end   = int(np.floor(float(mpmath.siegeltheta(t_max)) / np.pi))
    gram    = np.array([float(mpmath.grampoint(n)) for n in range(n_start, n_end+1)])
    Z       = np.array([float(mpmath.siegelz(g)) for g in gram])
    return gram, Z

def build_features(deviations, Z_gram, k=20, window=10):
    X, y = [], []
    for i in range(max(k, window), min(len(deviations)-1, len(Z_gram)-window-1)):
        dev_features = deviations[i-k:i]
        z_features   = Z_gram[i-window:i+window+1]
        X.append(np.concatenate([dev_features, z_features]))
        y.append(deviations[i])
    return np.array(X), np.array(y)

# Chargement
print("Chargement zeros_14.dat...")
zeros_14            = parse_zeros_dat(r"C:\Users\Tangu\Zeta Riemann\data\zeros_14.dat")
gram_14, Z_gram_14  = compute_gram_and_Z(14, 5000)
n14                 = min(len(zeros_14), len(gram_14))
dev_14              = zeros_14[:n14] - gram_14[:n14]

print("Chargement zeros_5000.dat...")
zeros_5000              = parse_zeros_dat(r"C:\Users\Tangu\Zeta Riemann\data\zeros_5000.dat")
gram_5000, Z_gram_5000  = compute_gram_and_Z(5000, 26000)
n5000                   = min(len(zeros_5000), len(gram_5000))
dev_5000                = zeros_5000[:n5000] - gram_5000[:n5000]

print(f"\nzeros_14   : {n14:,} zéros")
print(f"zeros_5000 : {n5000:,} zéros")

Chargement zeros_14.dat...
Chargement zeros_5000.dat...

zeros_14   : 4,520 zéros
zeros_5000 : 25,804 zéros


## 1. Z(t) aux points de Gram — une nouvelle feature

Les valeurs de Z aux points de Gram alternent en signe : moyenne +2 pour les pairs, -2 pour les impairs.  
Leur amplitude encode la structure locale de ζ — utile pour prédire où tombe le zéro dans l'intervalle.

Pour chaque intervalle de Gram n, on construit 41 features :
- 20 déviations passées $d_{n-20}, \ldots, d_{n-1}$
- 21 valeurs $Z(g_{n-10}), \ldots, Z(g_n), \ldots, Z(g_{n+10})$

In [2]:
# Valeurs Z aux points de Gram — propriété +2/-2
print(f"Moyenne Z aux Gram pairs   : {Z_gram_14[::2].mean():.4f}  (théorique : +2)")
print(f"Moyenne Z aux Gram impairs : {Z_gram_14[1::2].mean():.4f}  (théorique : −2)")
print(f"Std Z global               : {Z_gram_14.std():.4f}")
print(f"Min/Max Z                  : [{Z_gram_14.min():.2f}, {Z_gram_14.max():.2f}]")

# Distribution des déviations selon le range
print(f"\nDéviations zeros_14   — mean={dev_14.mean():.4f}  std={dev_14.std():.4f}")
print(f"Déviations zeros_5000 — mean={dev_5000.mean():.4f}  std={dev_5000.std():.4f}")
print(f"\n→ Les distributions diffèrent selon le range de t.")
print(f"  La std se réduit avec t : les zéros se rapprochent des points de Gram.")

Moyenne Z aux Gram pairs   : 2.0015  (théorique : +2)
Moyenne Z aux Gram impairs : -1.9985  (théorique : −2)
Std Z global               : 2.9998
Min/Max Z                  : [-14.75, 13.52]

Déviations zeros_14   — mean=-0.5517  std=0.3310
Déviations zeros_5000 — mean=-0.4069  std=0.2484

→ Les distributions diffèrent selon le range de t.
  La std se réduit avec t : les zéros se rapprochent des points de Gram.


## 2. Comparaison des feature sets

Trois feature sets testés sur le même dataset (zeros_5000, split 80/20) :
- **Déviations seules** : 20 déviations passées
- **Z seules** : 21 valeurs Z aux points de Gram environnants
- **Combiné** : 20 déviations + 21 valeurs Z = 41 features

In [3]:
k, window = 20, 10

# Features déviations seules
X_dev = np.array([dev_5000[i-k:i] for i in range(k, len(dev_5000)-1)])
y_dev = dev_5000[k+1:]

# Features Z seules
X_z = np.array([Z_gram_5000[i-window:i+window+1]
                for i in range(window, len(Z_gram_5000)-window-1)])
y_z = dev_5000[window:len(X_z)+window]
min_len_z = min(len(X_z), len(y_z))
X_z, y_z = X_z[:min_len_z], y_z[:min_len_z]

# Features combinées
X_comb, y_comb = build_features(dev_5000, Z_gram_5000, k, window)

results = {}
for name, X, y in [
    ("Déviations seules (20)", X_dev, y_dev),
    ("Z seules (21)",          X_z,   y_z),
    ("Combiné (41)",           X_comb, y_comb)
]:
    split   = int(0.8 * len(X))
    X_tr, X_te = X[:split], X[split:]
    y_tr, y_te = y[:split], y[split:]
    baseline = np.mean(np.abs(y_te - y_te.mean()))
    model = Ridge(alpha=1.0)
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = np.mean(np.abs(y_te - pred))
    results[name] = {"mae": mae, "baseline": baseline, "improvement": (baseline-mae)/baseline*100}
    print(f"{name:25s} — MAE={mae:.4f}  baseline={baseline:.4f}  +{(baseline-mae)/baseline*100:.1f}%")

Déviations seules (20)    — MAE=0.1522  baseline=0.1901  +19.9%
Z seules (21)             — MAE=0.1916  baseline=0.1900  +-0.8%
Combiné (41)              — MAE=0.1530  baseline=0.1900  +19.5%


In [4]:
# Importance des features sur le modèle combiné
split   = int(0.8 * len(X_comb))
X_tr, X_te = X_comb[:split], X_comb[split:]
y_tr, y_te = y_comb[:split], y_comb[split:]
model_comb = Ridge(alpha=1.0)
model_comb.fit(X_tr, y_tr)
pred_comb  = model_comb.predict(X_te)
errors     = y_te - pred_comb

labels = [f"dev_lag{i+1}" for i in range(20)] + [f"Z_g{i-10:+d}" for i in range(21)]
coeffs = model_comb.coef_

fig = go.Figure(go.Bar(
    x=labels, y=np.abs(coeffs),
    marker_color=[COLOR_MAIN if 'dev' in l else COLOR_ACCENT for l in labels]
))
fig.update_layout(
    template=TEMPLATE, height=420,
    title="Importance des features — |coefficient| Ridge (violet=déviations, jaune=Z)",
    xaxis_title="Feature", yaxis_title="|Coefficient|",
    xaxis_tickangle=-45
)
fig.show()

top5 = np.argsort(np.abs(coeffs))[::-1][:5]
print("Top 5 features :")
for i in top5:
    print(f"  {labels[i]:15s} : {coeffs[i]:+.4f}")

Top 5 features :
  dev_lag7        : +0.7978
  dev_lag17       : -0.7852
  dev_lag18       : -0.7822
  dev_lag6        : +0.7800
  dev_lag8        : +0.7208


## 3. Précision des prédictions

On visualise les prédictions vs réalité et on mesure la fiabilité du modèle combiné.

In [5]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Prédictions vs réalité", "Distribution des erreurs"))

x_axis = np.arange(len(y_te))
fig.add_trace(go.Scatter(
    x=x_axis, y=y_te,
    name="Déviation réelle",
    line=dict(color=COLOR_ACCENT, width=1)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=x_axis, y=pred_comb,
    name="Prédiction Ridge",
    line=dict(color=COLOR_MAIN, width=1, dash="dash")
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=errors, nbinsx=80,
    marker_color=COLOR_MAIN, opacity=0.85,
    name="Erreur"
), row=1, col=2)
fig.add_vline(x=0, line_color=COLOR_ACCENT, line_dash="dot", row=1, col=2)

fig.update_layout(template=TEMPLATE, height=420,
    title="Ridge combiné — précision sur zeros_5000")
fig.show()

print(f"Précision du modèle combiné :")
print(f"  MAE              : {np.mean(np.abs(errors)):.6f}")
print(f"  % erreurs < 0.1  : {(np.abs(errors) < 0.1).mean()*100:.1f}%")
print(f"  % erreurs < 0.2  : {(np.abs(errors) < 0.2).mean()*100:.1f}%")
print(f"  % erreurs < 0.5  : {(np.abs(errors) < 0.5).mean()*100:.1f}%")

Précision du modèle combiné :
  MAE              : 0.153032
  % erreurs < 0.1  : 39.0%
  % erreurs < 0.2  : 69.0%
  % erreurs < 0.5  : 99.5%


## 4. Limite : le modèle ne généralise pas entre ranges

Quand on entraîne sur zeros_14 et teste sur zeros_5000, les performances s'effondrent.  
La distribution des déviations change avec t — std plus faible, moyenne décalée.

C'est un **overfitting distribitionnel** — le modèle apprend les patterns d'un range de t  
qui ne se transfèrent pas à un autre range.

In [6]:
# Entraîner sur zeros_14, tester sur zeros_5000
X_14, y_14 = build_features(dev_14, Z_gram_14, k, window)
X_5000_full, y_5000_full = build_features(dev_5000, Z_gram_5000, k, window)

model_cross = Ridge(alpha=1.0)
model_cross.fit(X_14, y_14)
pred_cross = model_cross.predict(X_5000_full)

mae_cross    = np.mean(np.abs(y_5000_full - pred_cross))
baseline_5000 = np.mean(np.abs(y_5000_full - dev_5000.mean()))

print("Généralisation inter-ranges :")
print(f"  Entraîné sur zeros_14   : MAE train = {np.mean(np.abs(y_14 - model_cross.predict(X_14))):.4f}")
print(f"  Testé sur zeros_5000    : MAE test  = {mae_cross:.4f}")
print(f"  Baseline zeros_5000     : {baseline_5000:.4f}")
print(f"  → Le modèle fait PIRE que la baseline ({(mae_cross-baseline_5000)/baseline_5000*100:.1f}%)")
print(f"\nCause : distributions différentes selon le range de t")
print(f"  zeros_14   mean={dev_14.mean():.4f}  std={dev_14.std():.4f}")
print(f"  zeros_5000 mean={dev_5000.mean():.4f}  std={dev_5000.std():.4f}")

Généralisation inter-ranges :
  Entraîné sur zeros_14   : MAE train = 0.1623
  Testé sur zeros_5000    : MAE test  = 0.3273
  Baseline zeros_5000     : 0.2008
  → Le modèle fait PIRE que la baseline (63.0%)

Cause : distributions différentes selon le range de t
  zeros_14   mean=-0.5517  std=0.3310
  zeros_5000 mean=-0.4069  std=0.2484


## 5. Classification des intervalles — signe de Z comme prédicteur

Un intervalle de Gram $[g_n, g_{n+1}]$ est **bon** (exactement 1 zéro) quand Z change de signe entre ses deux bords.  
Si Z(g_n) et Z(g_{n+1}) ont le même signe, l'intervalle est **mauvais** (0 ou 2 zéros).

Cette règle simple donne une précision de **99.9%** — sans aucun ML.

In [7]:
# Classifier bons/mauvais intervalles via le signe de Z
zeros_per_gram = np.array([
    np.sum((zeros_14 >= gram_14[i]) & (zeros_14 < gram_14[i+1]))
    for i in range(len(gram_14)-1)
])
labels     = (zeros_per_gram == 1).astype(int)
sign_left  = np.sign(Z_gram_14[:-1])
sign_right = np.sign(Z_gram_14[1:])
sign_change = (sign_left != sign_right).astype(int)

acc = (sign_change == labels).mean()
print(f"Précision du signe de Z pour classifier bon/mauvais : {acc*100:.1f}%")
print(f"\nMatrice de confusion :")
print(f"  Changement de signe + bon   : {((sign_change==1) & (labels==1)).sum():,}")
print(f"  Changement de signe + mauvais: {((sign_change==1) & (labels==0)).sum():,}")
print(f"  Même signe + bon            : {((sign_change==0) & (labels==1)).sum():,}")
print(f"  Même signe + mauvais        : {((sign_change==0) & (labels==0)).sum():,}")

Précision du signe de Z pour classifier bon/mauvais : 99.9%

Matrice de confusion :
  Changement de signe + bon   : 3,882
  Changement de signe + mauvais: 3
  Même signe + bon            : 0
  Même signe + mauvais        : 634


## 6. Pipeline hybride — classification + régression

On combine les deux étapes :
1. **Classifier** l'intervalle via le signe de Z — 99.9% de précision
2. **Prédire la déviation** avec Ridge combiné uniquement sur les bons intervalles

En filtrant les mauvais intervalles, le modèle se concentre sur un problème homogène — **+33% d'amélioration** vs baseline, notre meilleur résultat.

In [8]:
# Ridge sur bons intervalles uniquement
X_comb, y_comb = build_features(dev_14, Z_gram_14, k=20, window=10)
good_mask = (zeros_per_gram == 1)
min_len   = min(len(good_mask), len(X_comb))
good_idx  = np.where(good_mask[:min_len])[0]
good_idx  = good_idx[(good_idx >= 20) & (good_idx < min_len)]

X_good = X_comb[good_idx - 20]
y_good = y_comb[good_idx - 20]

split = int(0.8 * len(X_good))
X_tr, X_te = X_good[:split], X_good[split:]
y_tr, y_te = y_good[:split], y_good[split:]

model_good = Ridge(alpha=1.0)
model_good.fit(X_tr, y_tr)
pred_good  = model_good.predict(X_te)

mae_good      = np.mean(np.abs(y_te - pred_good))
baseline_good = np.mean(np.abs(y_te - y_te.mean()))
errors_good   = y_te - pred_good

# Accuracy end-to-end
gaps_14      = np.diff(zeros_14)
rho_pred     = gram_14[good_idx[split:]] + pred_good
rho_real     = zeros_14[good_idx[split:]]
err_position = np.abs(rho_real - rho_pred)

print("Pipeline hybride — résultats :")
print(f"  Couverture (bons intervalles) : {labels.sum()/len(labels)*100:.1f}%")
print(f"  MAE déviation  : {mae_good:.4f}  (+{(baseline_good-mae_good)/baseline_good*100:.1f}% vs baseline)")
print(f"\nAccuracy position absolue du zéro :")
print(f"  MAE position          : {err_position.mean():.4f}")
print(f"  MAE / espacement moyen: {err_position.mean()/gaps_14.mean()*100:.1f}%")
print(f"  % zéros à ±0.2        : {(err_position < 0.2).mean()*100:.1f}%")
print(f"  % zéros à ±0.5        : {(err_position < 0.5).mean()*100:.1f}%")
print(f"  % zéros à ±1.0        : {(err_position < 1.0).mean()*100:.1f}%")

Pipeline hybride — résultats :
  Couverture (bons intervalles) : 85.9%
  MAE déviation  : 0.1629  (+23.2% vs baseline)

Accuracy position absolue du zéro :
  MAE position          : 0.1629
  MAE / espacement moyen: 14.8%
  % zéros à ±0.2        : 67.8%
  % zéros à ±0.5        : 98.4%
  % zéros à ±1.0        : 100.0%


## Ce qu'on a découvert

| Feature set | Amélioration vs baseline |
|---|---|
| **Déviations seules** | +21% |
| **Z(g_n) seules** | −1% (inutile seul) |
| **Combiné (41 features)** | +25% |
| **Combiné — bons intervalles seulement** | **+33%** |

### Pipeline final

```
1. Signe de Z(g_n) et Z(g_{n+1}) → intervalle bon ou mauvais (99.9% précis)
         ↓ Bon (85.9% des cas)              ↓ Mauvais (14.1%)
2. Ridge combiné → déviation prédite     → hors scope
3. ρ_n ≈ g_n + déviation
```

**Accuracy end-to-end** : 78.9% des zéros localisés à ±0.5, MAE = 27.7% de l'espacement moyen.

### Leçon clé

La classification mathématique (signe de Z) était plus puissante que tout le ML qu'on a essayé.  
Le ML apporte de la valeur en aval — prédire *où* dans l'intervalle tombe le zéro — mais pas en amont.  
C'est un résultat honnête : **la mathématique d'abord, le ML pour raffiner.**

---

### Tableau récapitulatif complet — notebooks 06, 07, 08

| Notebook | Modèle | Feature | Amélioration |
|---|---|---|---|
| 06 | Baseline | — | 0% |
| 06 | Ridge | Gaps bruts | +8% |
| 06 | LSTM | Gaps bruts | −10% |
| 07 | Ridge | Déviations | +21% |
| 07 | LSTM | Déviations | +20% |
| 08 | Ridge | Z seules | −1% |
| 08 | Ridge | Combiné | +25% |
| **08** | **Pipeline hybride** | **Signe Z + Combiné** | **+33%** |